# ATP Tennis Analysis

This notebook evaluates the three research goals:

1. How important is ranking for match outcomes?
2. Do players perform differently depending on the surface?
3. Which player characteristics are associated with winning matches?

The analysis uses:
- `data/matches_clean.csv`
- `data/stats_clean.csv`
- `data/top20_players_raw.csv`

In [141]:
import pandas as pd
import numpy as np

# import of CSV files
matches = pd.read_csv("data/matches_clean.csv")
stats = pd.read_csv("data/stats_clean.csv")
players = pd.read_csv("data/top20_players_raw.csv")

## Data quality inspection
Before proceeding to the main analysis, we inspect the cleaned data for any remaining issues:
- verify the number of missing values
- check for duplicate matches
- identify any rows with incomplete information

This ensures the dataset is reliable for the research questions.

In [142]:
# Basic data checks
print("\nMatches columns:", matches.columns.tolist())
print("Stats columns:", stats.columns.tolist())
print("Players columns:", players.columns.tolist())

print("\nMatches dtypes:")
print(matches.dtypes.to_string())

print("\nMissing values in matches:")
print(matches.isna().sum())


Matches columns: ['date', 'tournament', 'surface', 'Round', 'winner', 'winner_rank_at_match', 'loser', 'loser_rank_at_match', 'score', 'source_player']
Stats columns: ['player', 'surface', 'match_record', 'match_wins', 'match_losses', 'match_pct', 'tiebreak_record', 'tiebreak_wins', 'tiebreak_losses', 'tiebreak_pct', 'ace_rate_pct', 'first_serve_in_pct', 'first_serve_points_won_pct', 'second_serve_points_won_pct', 'service_games_held_pct', 'service_points_won_pct', 'return_games_won_pct', 'return_points_won_pct', 'total_points_won_pct', 'dominance_ratio']
Players columns: ['name', 'age', 'current_rank']

Matches dtypes:
date                        str
tournament                  str
surface                     str
Round                       str
winner                      str
winner_rank_at_match    float64
loser                       str
loser_rank_at_match     float64
score                       str
source_player               str

Missing values in matches:
date                   

In [143]:
missing = matches[matches["winner_rank_at_match"].isna() | matches["loser_rank_at_match"].isna()]

print("Total missing rows:", len(missing))
print(missing[["date", "source_player", "winner", "loser", "score", "winner_rank_at_match", "loser_rank_at_match"]].head())

print("\nCounts by source_player:")
print(missing["source_player"].value_counts())

Total missing rows: 1
           date     source_player            winner     loser    score  \
786  6-Feb-2026  Alexander Bublik  Alexander Bublik  Hugo Nys  6-0 6-3   

     winner_rank_at_match  loser_rank_at_match  
786                  10.0                  NaN  

Counts by source_player:
source_player
Alexander Bublik    1
Name: count, dtype: int64


In [144]:
print("\nStats dtypes:")
print(stats.dtypes.to_string())


Stats dtypes:
player                           str
surface                          str
match_record                     str
match_wins                     int64
match_losses                   int64
match_pct                        str
tiebreak_record                  str
tiebreak_wins                  int64
tiebreak_losses                int64
tiebreak_pct                     str
ace_rate_pct                     str
first_serve_in_pct               str
first_serve_points_won_pct       str
second_serve_points_won_pct      str
service_games_held_pct           str
service_points_won_pct           str
return_games_won_pct             str
return_points_won_pct            str
total_points_won_pct             str
dominance_ratio                  str


In [152]:
print("\nMissing values in stats:")
print(stats.isna().sum())


Missing values in stats:
player                         0
surface                        0
match_record                   0
match_wins                     0
match_losses                   0
match_pct                      3
tiebreak_record                0
tiebreak_wins                  0
tiebreak_losses                0
tiebreak_pct                   6
ace_rate_pct                   0
first_serve_in_pct             0
first_serve_points_won_pct     0
second_serve_points_won_pct    0
service_games_held_pct         0
service_points_won_pct         0
return_games_won_pct           0
return_points_won_pct          0
total_points_won_pct           0
dominance_ratio                0
dtype: int64


In [151]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None, 'display.expand_frame_repr', False):
    print(stats[stats.isna().any(axis=1)].to_string())

             player surface match_record  match_wins  match_losses match_pct tiebreak_record  tiebreak_wins  tiebreak_losses tiebreak_pct ace_rate_pct first_serve_in_pct first_serve_points_won_pct second_serve_points_won_pct service_games_held_pct service_points_won_pct return_games_won_pct return_points_won_pct total_points_won_pct dominance_ratio
18  Daniil Medvedev    Clay    4-4 (50%)           4             4       50%         0-0 (-)              0                0          NaN         6.6%              60.1%                      71.2%                       48.7%                  76.9%                  62.2%                25.3%                 38.7%                50.3%            1.02
35  Lorenzo Musetti   Grass     0-1 (0%)           0             1        0%         0-0 (-)              0                0          NaN         5.6%              58.9%                      61.9%                       61.4%                  72.2%                  61.7%                 5.3%       

In [153]:
print("\nDuplicate match rows:", matches.duplicated(
   subset=["date", "tournament", "surface", "Round", "winner", "loser", "score"]
).sum())


Duplicate match rows: 132


In [154]:
# Inspect whether duplicates are exact copies, just from different sources (2 players)
exact_duplicate_counts = duplicates.groupby(
    ["date", "tournament", "surface", "Round", "winner", "loser", "score"]
).size().sort_values(ascending=False).tail(30)

print("\nTop duplicate groups by count:")
print(exact_duplicate_counts)


Top duplicate groups by count:
date        tournament            surface  Round  winner            loser             score            
4-Mar-2026  Indian Wells Masters  Hard     R32    Learner Tien      Ben Shelton       7-6(3) 4-6 6-3       2
                                           SF     Daniil Medvedev   Carlos Alcaraz    6-3 7-6(3)           2
                                                  Jannik Sinner     Alexander Zverev  6-2 6-4              2
5-Jan-2026  Hong Kong             Hard     F      Alexander Bublik  Lorenzo Musetti   7-6(2) 6-3           2
            United Cup            Hard     RR     Jakub Mensik      Casper Ruud       7-5 7-6(6)           2
7-Aug-2025  Cincinnati Masters    Hard     F      Carlos Alcaraz    Jannik Sinner     5-0 RET              2
                                           QF     Alexander Zverev  Ben Shelton       6-2 6-2              2
                                           R16    Ben Shelton       Jiri Lehecka      6-4 6-4        

In [155]:
matches = matches_unique.copy()  # Use the deduplicated version for analysis

### Note

The inspection shows only one row remains with a missing opponent rank.

Duplicate match groups were also inspected, and the duplicate entries are repeated scrapes of the same match. For the purposes of the match-level analysis, the cleaned dataset will be treated as one entry per match.

The remaining missing row corresponds to an unranked / unclassified opponent on the source site, so it is valid to treat it as missing and exclude it from rank-difference analysis.

This means the cleaned dataset is otherwise complete and ready for the main analysis.

## Data validation

We begin by checking:
- check rank columns for missing values
- verify no duplicate match rows remain
- confirm stats percentage fields are still strings and need parsing

In [156]:
print("\nFinal missing values:")
print(matches.isna().sum())

print("\nDuplicate rows after validation:", matches.duplicated(
    subset=["date", "tournament", "surface", "Round", "winner", "loser", "score"]
).sum())


Final missing values:
date                    0
tournament              0
surface                 0
Round                   0
winner                  0
winner_rank_at_match    0
loser                   0
loser_rank_at_match     1
score                   0
source_player           0
dtype: int64

Duplicate rows after validation: 0


## Analytical dataset construction


In [157]:
# Parse stats percentages
percentage_columns = [
    "match_pct", "tiebreak_pct", "ace_rate_pct", "first_serve_in_pct", "first_serve_points_won_pct",
    "second_serve_points_won_pct", "service_games_held_pct", "service_points_won_pct",
    "return_games_won_pct", "return_points_won_pct", "total_points_won_pct",
    "dominance_ratio"
]

for col in percentage_columns:
    if col in stats.columns:
        stats[col] = pd.to_numeric(
            stats[col].replace("-", np.nan).astype(str).str.rstrip("%"),
            errors="coerce",
        )

stats.head(10)

,player,surface,match_record,match_wins,match_losses,match_pct,tiebreak_record,tiebreak_wins,tiebreak_losses,tiebreak_pct,ace_rate_pct,first_serve_in_pct,first_serve_points_won_pct,second_serve_points_won_pct,service_games_held_pct,service_points_won_pct,return_games_won_pct,return_points_won_pct,total_points_won_pct,dominance_ratio
0,Jannik Sinner,Overall last 52,72-8 (90%),72,8,90.0,19-5 (79%),19,5,79.0,10.8,63.7,79.8,58.2,92.2,71.9,32.3,42.2,56.3,1.51
1,Jannik Sinner,Hard,51-5 (91%),51,5,91.0,15-1 (94%),15,1,94.0,12.5,64.3,81.3,57.4,93.1,72.8,31.5,41.7,56.5,1.53
2,Jannik Sinner,Clay,13-2 (87%),13,2,87.0,3-4 (43%),3,4,43.0,5.5,61.6,74.5,59.3,88.2,68.7,38.7,45.2,56.3,1.44
3,Jannik Sinner,Grass,8-1 (89%),8,1,89.0,1-0 (100%),1,0,100.0,11.0,63.6,80.2,60.3,93.5,72.9,26.4,40.1,55.6,1.48
4,Carlos Alcaraz,Overall last 52,70-7 (91%),70,7,91.0,20-13 (61%),20,13,61.0,7.5,65.5,74.7,57.1,89.1,68.6,30.1,41.4,54.5,1.32
5,Carlos Alcaraz,Hard,40-5 (89%),40,5,89.0,10-7 (59%),10,7,59.0,7.7,66.1,75.9,57.6,90.6,69.7,30.0,41.2,54.7,1.36
6,Carlos Alcaraz,Clay,19-1 (95%),19,1,95.0,8-2 (80%),8,2,80.0,3.5,67.2,69.5,56.7,84.1,65.3,36.2,44.7,54.7,1.29
7,Carlos Alcaraz,Grass,11-1 (92%),11,1,92.0,2-4 (33%),2,4,33.0,12.4,61.6,78.9,56.5,91.4,70.3,22.6,37.5,53.6,1.26
8,Novak Djokovic,Overall last 52,34-7 (83%),34,7,83.0,8-8 (50%),8,8,50.0,9.3,66.2,76.3,55.4,87.8,69.2,26.1,39.7,54.0,1.29
9,Novak Djokovic,Hard,20-4 (83%),20,4,83.0,5-4 (56%),5,4,56.0,9.3,65.4,75.8,56.0,87.6,68.9,24.0,38.2,53.4,1.23


## 1. Ranking and match outcomes

We test whether ranking difference predicts match results and how often upsets occur.

In [47]:
rank_bins = pd.cut(
    analysis_df["rank_difference"],
    bins=[-100, -10, -5, -1, 0, 5, 10, 100],
    labels=[
        "winner_much_favored (-10 lower rank)",
        "winner_favored (-5 to -10)",
        "winner_close (-1 to -5)",
        "winner_slightly_favored (0 to -1)",
        "loser_slightly_favored (0 to 1)",
        "loser_favored (5 to 10)",
        "loser_much_favored (10+ lower rank)",
    ],
)

rank_summary = (
    analysis_df.assign(rank_bin=rank_bins)
    .groupby("rank_bin", observed=True)
    .agg(
        matches=("rank_difference", "size"),
        upsets=("upset", "sum"),
        avg_rank_diff=("rank_difference", "mean"),
    )
    .reset_index()
)
rank_summary["upset_rate"] = rank_summary["upsets"] / rank_summary["matches"]
rank_summary.sort_values("avg_rank_diff")

,rank_bin,matches,upsets,avg_rank_diff,upset_rate
0,winner_much_favored (-10 lower rank),36,36,-35.638889,1.0
1,winner_favored (-5 to -10),7,7,-6.714286,1.0
2,winner_close (-1 to -5),19,19,-2.000000,1.0
3,loser_slightly_favored (0 to 1),46,0,3.021739,0.0
4,loser_favored (5 to 10),18,0,8.111111,0.0
5,loser_much_favored (10+ lower rank),188,0,44.287234,0.0


### Interpretation

If the upset rate is high when the winner is lower-ranked, then ranking is less predictive.
If most wins happen when the winner is strongly favored, ranking is more important.

## 2. Surface-specific performance

We compare player performance by surface using match-level outcomes and surface stats.

In [5]:
# Build match records by surface
win_loss_frames = []
win_loss_frames.append(
    analysis_df[["winner", "surface"]].assign(player=analysis_df["winner"], result="W")
)
win_loss_frames.append(
    analysis_df[["loser", "surface"]].assign(player=analysis_df["loser"], result="L")
)
surface_results = pd.concat(win_loss_frames, ignore_index=True)

surface_records = (
    surface_results.groupby(["player", "surface", "result"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
surface_records["matches"] = surface_records.get("W", 0) + surface_records.get("L", 0)
surface_records["win_rate"] = surface_records.get("W", 0) / surface_records["matches"]

print("Surface-specific win rates (top 10 by matches):")
surface_records.sort_values("matches", ascending=False).head(10)

Surface-specific win rates (top 10 by matches):


result,player,surface,L,W,matches,win_rate
101,Jannik Sinner,Hard,8,58,66,0.878788
20,Alexander Zverev,Hard,24,39,63,0.619048
66,Daniil Medvedev,Hard,16,41,57,0.719298
50,Carlos Alcaraz,Hard,7,46,53,0.867925
143,Novak Djokovic,Hard,6,22,28,0.785714
48,Carlos Alcaraz,Clay,1,20,21,0.952381
18,Alexander Zverev,Clay,5,15,20,0.750000
99,Jannik Sinner,Clay,4,13,17,0.764706
49,Carlos Alcaraz,Grass,2,11,13,0.846154
141,Novak Djokovic,Clay,3,10,13,0.769231


### Interpretation

The surface comparison shows whether individual players are stronger on Hard, Clay, or Grass.
If a player has a much better win rate or serve/return profile on one surface, that supports surface-specific performance differences.

## 3. Player characteristics and winning

We evaluate whether service and return metrics are associated with match wins.

In [6]:
# Average performance gaps by upset
gap_summary = (
    analysis_df[["upset"] + [f"{col}_gap" for col in gap_features]]
    .groupby("upset")
    .mean()
    .reset_index()
)
gap_summary.columns = ["upset"] + [f"{col}_gap_mean" for col in gap_features]
print("Performance gaps (positive = winner better):")
print(gap_summary.T)

print("\n\nCorrelation of performance gaps with rank difference:")
correlations = {}
for col in [f"{col}_gap" for col in gap_features]:
    if col in analysis_df.columns:
        correlations[col] = analysis_df[col].corr(analysis_df["rank_difference"])
pd.Series(correlations).sort_values(ascending=False)

Performance gaps (positive = winner better):
                                             0         1
upset                                    False      True
service_points_won_pct_gap_mean       0.022608 -0.014257
return_points_won_pct_gap_mean        0.042962 -0.002714
ace_rate_pct_gap_mean                -0.008861  0.003571
first_serve_points_won_pct_gap_mean   0.020608 -0.005429
second_serve_points_won_pct_gap_mean  0.026671 -0.016057


Correlation of performance gaps with rank difference:


second_serve_points_won_pct_gap    0.441025
return_points_won_pct_gap          0.436930
service_points_won_pct_gap         0.406345
first_serve_points_won_pct_gap     0.103847
ace_rate_pct_gap                  -0.320697
dtype: float64

### Interpretation

- Positive average gaps show winners usually outperform losers on those metrics.
- If the gap is especially large in non-upsets, those metrics may reflect expected dominance.
- Correlation to rank difference shows whether better serve/return stats are aligned with ranking advantage.

## Conclusions

### Key Findings

1. **Ranking is important** but not deterministic—upset frequency varies by rank gap
2. **Surface matters**—players show different win rates across Hard/Clay/Grass
3. **Second serve and return strength** are the strongest predictors of ranking advantage

### Next Steps

- Add visualizations (upset rates by rank gap, win rates by surface)
- Fit a logistic regression model on `upset` outcome
- Compare surface-specific stats vs overall performance by player
- Export tables and plots for the final report